## Gold Layer: Analytical Data Views

This notebook generates business-level aggregations and metric views from the cleaned Silver data (`cardio_sil`).

### Key Objectives:
* **Cohort Analysis:** Measure cardiovascular disease rates across age groups, gender, and lifestyle factors.
* **Risk Stratification:** Analyze risk distribution across blood pressure, cholesterol, and glucose categories.
* **Dashboard Ready:** Create clean, aggregated tables for reporting and Power BI dashboards.

In [0]:
df=spark.table('cardio_sil')
display(df)

id,gender,height,weight,systolic_bp,diastolic_bp,cholesterol,gluc,smoke,alco,active,cardio,age_years,BMI,BP_Category
24,2,172,112.0,120,80,1,1,0,0,0,1,46.0,37.85830178474852,Hypertension
30,2,163,83.0,120,80,1,1,0,0,1,0,54.0,31.239414355075468,Hypertension
31,1,157,69.0,130,80,1,1,0,0,1,0,59.0,27.993022029291247,Hypertension
44,1,159,60.0,110,70,1,1,0,0,1,0,40.0,23.73323840037973,Normal
45,1,160,73.0,130,85,1,1,0,0,0,1,57.0,28.515624999999993,Hypertension
57,2,171,69.0,140,90,1,1,0,0,1,1,59.0,23.59700420642249,Hypertension
66,1,169,84.0,150,100,1,1,0,0,1,1,50.0,29.410734918245165,Hypertension
88,2,170,69.0,130,80,1,1,0,0,1,0,52.0,23.87543252595156,Hypertension
109,1,159,49.0,120,70,1,1,0,0,1,0,45.0,19.382144693643447,Elevated
121,1,151,92.0,130,90,1,1,0,0,0,1,64.0,40.349107495285295,Hypertension


In [0]:
display(df.summary())

summary,id,gender,height,weight,systolic_bp,diastolic_bp,cholesterol,gluc,smoke,alco,active,cardio,age_years,BMI,BP_Category
count,68669,68669,68669,68669,68669,68669,68669,68669,68669,68669,68669,68669,68669,68669,68669
mean,49975.892935676944,1.3486289300849,164.3609925876305,74.11907119660984,126.66778313358284,81.30143150475469,1.3646186780060872,1.22567679739038,0.08797273879042945,0.0533574101850908,0.8033464882261282,0.4946919279441961,53.326974326115135,27.521860472335884,null
stddev,28844.850985543577,0.4765397215530436,8.183872715708517,14.33328015321333,16.682908981872185,9.424596954050722,0.6788703822701555,0.5715493902742119,0.2832573113743155,0.22474681874823718,0.39747101621255193,0.4999754640647272,6.768009815077785,6.052176110616944,null
min,0,1,55,11.0,60,40,1,1,0,0,0,0,30.0,3.471783865673526,Elevated
25%,25006,1,159,65.0,120,80,1,1,0,0,1,0,48.0,23.875114784205696,null
50%,50006,1,165,72.0,120,80,1,1,0,0,1,0,54.0,26.346494034400994,null
75%,74862,2,170,82.0,140,90,1,1,0,0,1,1,58.0,30.119375573921033,null
max,99999,2,250,200.0,240,150,3,3,1,1,1,1,65.0,298.6666666666667,Normal


Risk profile by age group


In [0]:
from pyspark.sql.functions import when, col
df_gold=df.withColumn("age_group",
    when((col("age_years") >= 30) & (col("age_years") < 40), "30-40")
    .when((col("age_years") >= 40) & (col("age_years") < 50), "40-50")
    .when((col("age_years") >= 50) & (col("age_years") < 60), "50-60")
    .otherwise("60+")
)

In [0]:
from pyspark.sql.functions import *
risk_profile=df_gold.groupBy("age_group", "cholesterol") \
.agg(
        avg("cardio").alias("avg_cardio_risk"),
        avg("BMI").alias("avg_BMI")
    )

In [0]:
display(risk_profile)

age_group,cholesterol,avg_cardio_risk,avg_BMI
40-50,1,0.3078430047963251,26.46595487015521
50-60,1,0.44336920337539676,27.1691453996302
60+,1,0.6037431122933689,27.48980161168382
50-60,3,0.7344521224086871,29.40425315770479
60+,2,0.7077687443541102,28.884215068712372
50-60,2,0.5749633046760327,28.622576157563913
40-50,3,0.7320872274143302,29.206248132386683
60+,3,0.813953488372093,29.844647972881404
40-50,2,0.5377483443708609,28.066117249782778
30-40,1,0.16913946587537093,26.57047428765639


In [0]:
risk_profile.write.mode('overwrite').saveAsTable('risk_profile')

Lifestyle risk matrix of smoker

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW gold_lifestyle_risk_matrix AS
SELECT smoke,
    CASE 
        WHEN smoke = 1 THEN 'Smoker'
        ELSE 'Non-Smoker'
    END AS smoking_status,

    AVG(systolic_bp) AS avg_systolic_bp,
    AVG(diastolic_bp) AS avg_diastolic_bp,
    AVG(BMI) AS avg_BMI,
    AVG(cardio) AS cardio_risk_rate

FROM workspace.default.cardio_sil
GROUP BY smoke;""")


DataFrame[]

In [0]:
display(spark.sql("SELECT * FROM gold_lifestyle_risk_matrix"))

smoke,smoking_status,avg_systolic_bp,avg_diastolic_bp,avg_BMI,cardio_risk_rate
0,Non-Smoker,126.52331225649868,81.22695918758383,27.574269672737383,0.4972216899789232
1,Smoker,128.16553550736634,82.07349776527064,26.97852602394167,0.4684654858467141


In [0]:
lifestyle_risk_matrix_smoker=spark.sql("SELECT * FROM gold_lifestyle_risk_matrix")
lifestyle_risk_matrix_smoker.write.mode('overwrite').saveAsTable('risk_matrix_smoker')

BMI category analysis

In [0]:
spark.sql("""CREATE OR REPLACE VIEW gold_bmi_category AS
SELECT CASE 
        WHEN BMI < 18.5 THEN 'Underweight'
        WHEN BMI BETWEEN 18.5 AND 25 THEN 'Normal'
        WHEN BMI BETWEEN 25 AND 30 THEN 'Overweight'
        ELSE 'Obese'
    END AS bmi_category,
    AVG(cardio) AS cardio_risk,
    AVG(systolic_bp) AS avg_bp

FROM workspace.default.cardio_sil
GROUP BY 
    CASE 
        WHEN BMI < 18.5 THEN 'Underweight'
        WHEN BMI BETWEEN 18.5 AND 25 THEN 'Normal'
        WHEN BMI BETWEEN 25 AND 30 THEN 'Overweight'
        ELSE 'Obese'
    END;""")

DataFrame[]

In [0]:
display(spark.sql("SELECT * FROM gold_bmi_category"))

bmi_category,cardio_risk,avg_bp
Obese,0.6246941045606229,132.76662958843158
Overweight,0.5055061156487464,126.94774269576172
Normal,0.397884970711955,122.36564846483469
Underweight,0.27527216174183516,115.6049766718507


In [0]:
gold_bmi_category=spark.sql('SELECT * FROM gold_bmi_category')
gold_bmi_category.write.mode('overwrite').saveAsTable('bmi_category')

Showing Cholesterol impact

In [0]:
spark.sql("""CREATE OR REPLACE VIEW gold_cholesterol_risk AS
SELECT cholesterol,
    AVG(cardio) AS cardio_risk,
    AVG(BMI) AS avg_BMI,
    AVG(systolic_bp) AS avg_bp

FROM workspace.default.cardio_sil
GROUP BY cholesterol;""")

DataFrame[]

In [0]:
display(spark.sql("SELECT * FROM gold_cholesterol_risk"))

cholesterol,cardio_risk,avg_BMI,avg_bp
1,0.43539805825242717,27.028643374689114,124.81405825242719
3,0.7626127843436269,29.544602037811792,133.840894649892
2,0.5963440860215053,28.541616079871616,130.8636559139785


In [0]:
cholesterol_risk=spark.sql('SELECT * FROM gold_cholesterol_risk')
cholesterol_risk.write.mode('overwrite').saveAsTable('cholesterol_risk')